<a href="https://colab.research.google.com/github/ParthBrijpuria/ai-diagnostic-companion/blob/main/diffusion_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diffusion Model on CIFAR-10
Task 1: Parth's implementation of an unconditional Diffusion model with a UNet architecture.


In [1]:
# Tweak: Use CIFAR-10 and set IMG_SIZE to 32 for the dataset.
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch import nn
import math
import matplotlib.pyplot as plt
import os

# Tweak: Configurable parameters for easily changing model scale and input resolution
IMG_SIZE = 32 # Changed to 32 for CIFAR-10
BATCH_SIZE = 512  # Increased for faster GPU processing
EPOCHS = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"



In [2]:
# Tweak: Changed dataset from Stanford Cars to CIFAR-10.
def load_transformed_dataset():
    data_transforms = [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), # Scales data into [0,1]
        transforms.Lambda(lambda t: (t * 2) - 1) # Scale between [-1, 1]
    ]
    data_transform = transforms.Compose(data_transforms)

    # Load CIFAR-10 dataset
    train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=data_transform)
    test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=data_transform)

    return torch.utils.data.ConcatDataset([train_dataset, test_dataset])

data = load_transformed_dataset()
dataloader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2, pin_memory=True) # Added workers & pin_memory



100%|██████████| 170M/170M [00:13<00:00, 12.2MB/s]


In [3]:
# Tweak: Make Unet flexible so layers, channels, etc. can be changed via simple parameters.
class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, up=False):
        super().__init__()
        self.time_mlp =  nn.Linear(time_emb_dim, out_ch)
        if up:
            self.conv1 = nn.Conv2d(2*in_ch, out_ch, 3, padding=1)
            self.transform = nn.ConvTranspose2d(out_ch, out_ch, 4, 2, 1)
        else:
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
            self.transform = nn.Conv2d(out_ch, out_ch, 4, 2, 1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bnorm1 = nn.BatchNorm2d(out_ch)
        self.bnorm2 = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU()

    def forward(self, x, t):
        h = self.bnorm1(self.relu(self.conv1(x)))
        time_emb = self.relu(self.time_mlp(t))
        time_emb = time_emb[(..., ) + (None, ) * 2]
        h = h + time_emb
        h = self.bnorm2(self.relu(self.conv2(h)))
        return self.transform(h)


class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class SimpleUnet(nn.Module):
    """
    A simplified Unet architecture.
    Tweak: Parameters can be easily changed when initializing the model instance.
    """
    def __init__(self, image_channels=3, down_channels=(64, 128, 256, 512), up_channels=(512, 256, 128, 64), out_dim=3, time_emb_dim=32):
        super().__init__()

        self.time_mlp = nn.Sequential(
                SinusoidalPositionEmbeddings(time_emb_dim),
                nn.Linear(time_emb_dim, time_emb_dim),
                nn.ReLU()
            )

        self.conv0 = nn.Conv2d(image_channels, down_channels[0], 3, padding=1)

        self.downs = nn.ModuleList([Block(down_channels[i], down_channels[i+1], time_emb_dim) \
                    for i in range(len(down_channels)-1)])

        self.ups = nn.ModuleList([Block(up_channels[i], up_channels[i+1], time_emb_dim, up=True) \
                    for i in range(len(up_channels)-1)])

        self.output = nn.Conv2d(up_channels[-1], out_dim, 1)

    def forward(self, x, timestep):
        t = self.time_mlp(timestep)
        x = self.conv0(x)
        residual_inputs = []
        for down in self.downs:
            x = down(x, t)
            residual_inputs.append(x)
        for up in self.ups:
            residual_x = residual_inputs.pop()
            x = torch.cat((x, residual_x), dim=1)
            x = up(x, t)
        return self.output(x)



In [6]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch import nn
import math
import matplotlib.pyplot as plt
import os

# Tweak: Implement diffusion as a class with forwardstep(), train(), and deploy() methods.
# The DL model (UNet) is passed into the constructor.
class Diffusion:
    def __init__(self, model, img_size=32, device="cuda", T=300):
        self.model = model
        self.img_size = img_size
        self.device = device
        self.T = T

        # Define beta schedule
        self.betas = self.linear_beta_schedule(timesteps=T).to(device)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, axis=0).to(device)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0).to(device)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas).to(device)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod).to(device)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod).to(device)
        self.posterior_variance = (self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)).to(device)

    def linear_beta_schedule(self, timesteps, start=0.0001, end=0.02):
        return torch.linspace(start, end, timesteps)

    def get_index_from_list(self, vals, t, x_shape):
        batch_size = t.shape[0]
        out = vals.gather(-1, t) # Removed .cpu() from t
        return out.reshape(batch_size, *((1,) * (len(x_shape) - 1))).to(self.device)

    def forwardstep(self, x_0, t):
        """ Tweak: forwardstep method as requested """
        noise = torch.randn_like(x_0).to(self.device)
        sqrt_alphas_cumprod_t = self.get_index_from_list(self.sqrt_alphas_cumprod, t, x_0.shape)
        sqrt_one_minus_alphas_cumprod_t = self.get_index_from_list(self.sqrt_one_minus_alphas_cumprod, t, x_0.shape)
        return sqrt_alphas_cumprod_t * x_0.to(self.device) + sqrt_one_minus_alphas_cumprod_t * noise, noise

    def get_loss(self, x_0, t):
        x_noisy, noise = self.forwardstep(x_0, t)
        noise_pred = self.model(x_noisy, t)
        return F.l1_loss(noise, noise_pred)

    @torch.no_grad()
    def sample_timestep(self, x, t):
        betas_t = self.get_index_from_list(self.betas, t, x.shape)
        sqrt_one_minus_alphas_cumprod_t = self.get_index_from_list(
            self.sqrt_one_minus_alphas_cumprod, t, x.shape
        )
        sqrt_recip_alphas_t = self.get_index_from_list(self.sqrt_recip_alphas, t, x.shape)

        # Call model (current image - noise prediction)
        model_mean = sqrt_recip_alphas_t * (
            x - betas_t * self.model(x, t) / sqrt_one_minus_alphas_cumprod_t
        )
        posterior_variance_t = self.get_index_from_list(self.posterior_variance, t, x.shape)

        if t == 0:
            return model_mean
        else:
            noise = torch.randn_like(x)
            return model_mean + torch.sqrt(posterior_variance_t) * noise

    @torch.no_grad()
    def deploy(self, num_images=10, save_folder="generated_samples"):
        """ Tweak: deploy method to generate images and save them in a folder """
        os.makedirs(save_folder, exist_ok=True)

        # Start from pure noise
        img = torch.randn((num_images, 3, self.img_size, self.img_size), device=self.device)

        for i in range(0, self.T)[::-1]:
            t = torch.full((num_images,), i, device=self.device, dtype=torch.long)
            img = self.sample_timestep(img, t)
            img = torch.clamp(img, -1.0, 1.0)

        # Reverse transforms to [0, 1] for saving
        img = (img + 1) / 2

        # Save images
        import torchvision.utils as vutils
        for idx in range(num_images):
            vutils.save_image(img[idx], os.path.join(save_folder, f"sample_{idx}.png"))

        print(f"Deployment complete. Saved {num_images} generated images to {save_folder}/")

        # Also show the images using matplotlib
        plt.figure(figsize=(15, 3))
        for idx in range(num_images):
            plt.subplot(1, num_images, idx+1)
            plt.imshow(img[idx].cpu().permute(1, 2, 0))
            plt.axis('off')
        plt.show()

        return img

    def train(self, dataloader, epochs=100, lr=0.001):
        """ Tweak: train method encapsulated within the class """
        self.model.to(self.device)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        scaler = torch.cuda.amp.GradScaler() # Added for Mixed Precision Training

        for epoch in range(epochs):
            for step, batch in enumerate(dataloader):
                optimizer.zero_grad()

                # Handling if dataloader returns a tuple (image, label)
                if isinstance(batch, list) or isinstance(batch, tuple):
                    images = batch[0].to(self.device)
                else:
                    images = batch.to(self.device)

                t = torch.randint(0, self.T, (images.shape[0],), device=self.device).long()

                # Mixed Precision Context
                with torch.cuda.amp.autocast():
                    loss = self.get_loss(images, t)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            print(f"Epoch {epoch} | Loss: {loss.item():.4f} ")

In [ ]:
# Tweak: Create a medium-sized Unet and pass it to the Diffusion class
# We keep the model simple to ensure it trains quickly while still generating images.
# Tweak: Reduced channels (32,64,128,256) for a faster model (from ~15M params to ~3.8M params)
unet_model = SimpleUnet(image_channels=3,
                        down_channels=(32, 64, 128, 256),
                        up_channels=(256, 128, 64, 32),
                        out_dim=3,
                        time_emb_dim=32)
print("Num params: ", sum(p.numel() for p in unet_model.parameters()))

# Instantiate the unconditional Diffusion model
diffusion_model = Diffusion(model=unet_model, img_size=IMG_SIZE, device=DEVICE, T=300)

# Run training
print("Starting training...")
diffusion_model.train(dataloader, epochs=EPOCHS, lr=0.001)

# Deploy and save generated sample images
print("Generating images...")
generated_images = diffusion_model.deploy(num_images=10, save_folder="generated_samples")



Num params:  3878147
Starting training...


/tmp/ipykernel_1253/4138689718.py:107: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # Added for Mixed Precision Training
/tmp/ipykernel_1253/4138689718.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 0 | Loss: 0.2564 
Epoch 1 | Loss: 0.2059 
Epoch 2 | Loss: 0.1925 
Epoch 3 | Loss: 0.1768 
Epoch 4 | Loss: 0.1830 
Epoch 5 | Loss: 0.1821 
Epoch 6 | Loss: 0.1751 
Epoch 7 | Loss: 0.1686 
Epoch 8 | Loss: 0.1708 
Epoch 9 | Loss: 0.1685 
Epoch 10 | Loss: 0.1649 
Epoch 11 | Loss: 0.1706 
Epoch 12 | Loss: 0.1682 
Epoch 13 | Loss: 0.1764 
Epoch 14 | Loss: 0.1623 
Epoch 15 | Loss: 0.1691 
Epoch 16 | Loss: 0.1630 
Epoch 17 | Loss: 0.1687 
Epoch 18 | Loss: 0.1715 
Epoch 19 | Loss: 0.1696 
